<a href="https://colab.research.google.com/github/spbromberg/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [13]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    # Keep track of the most recent retryable failure.
    last = None

    # Make up to `tries` attempts.
    for attempt in range(tries):
        try:
            # Request the data with a timeout.
            r = requests.get(BASE, params=params, timeout=15)

            # On success, return just the daily data as a DataFrame.
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

            # Stop immediately for statuses that should not be retried,
            # such as 400 or 404.
            if r.status_code not in RETRY_STATUSES:
                raise RuntimeError(
                    f'not retryable: {r.status_code} {r.text[:200]}'
                )

            # Save the status so the final error can report it.
            last = f'status {r.status_code}'

        # A timeout or connection problem has no response status code,
        # so record the exception and retry it too.
        except requests.RequestException as e:
            last = repr(e)

        # If this was the final attempt, stop without waiting again.
        if attempt == tries - 1:
            break

        # Wait longer after each failed attempt: 0.5 seconds, then 1 second.
        delay = 0.5 * (2 ** attempt)
        print(f'attempt {attempt + 1} failed ({last}); retrying in {delay}s')
        time.sleep(delay)

    # Raise if every attempt failed instead of returning an empty result.
    raise RuntimeError(f'gave up after {tries} attempts: {last}')

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [14]:
# Dates we want to compare
dates_to_keep = ['2026-09-05', '2026-10-03']
used_fallback = False

try:
    # Make one request for the full date range.
    api_weather = get_weather('2026-09-05', '2026-10-03')

    # Keep only the two weekend dates.
    wx_raw = api_weather[api_weather['time'].isin(dates_to_keep)].copy()

    # Treat a missing date as a failed request so we can use the sample data.
    if len(wx_raw) != 2:
        raise RuntimeError(f'expected 2 dates, got {len(wx_raw)}')

    print('got', len(wx_raw), 'daily weather rows')

except RuntimeError as e:
    # A non-retryable status means the request needs fixing; let that error show.
    if str(e).startswith('not retryable:'):
        raise

    # A request that failed after its retries can use the fallback sample.
    print('weather request failed:', e)
    used_fallback = True
    wx_raw = None

except requests.RequestException as e:
    # This handles a network error that reached this cell.
    print('network failed:', type(e).__name__)
    used_fallback = True
    wx_raw = None

got 2 daily weather rows


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [15]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# Use the sample rows only if the API request failed.
if used_fallback:
    wx_raw = FALLBACK.copy()
    wx_raw['source'] = 'fallback sample'
else:
    wx_raw['source'] = 'open-meteo'

wx_raw

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [16]:
# Rename the API fields to names used by our own table.
wx = wx_raw.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})

# Keep the required columns and make date a datetime value.
wx = wx[['date', 'temp_max', 'precip_mm', 'source']].copy()
wx['date'] = pd.to_datetime(wx['date'])

wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [17]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [18]:
# Find the rows with the most and least precipitation.
wettest = wx.loc[wx['precip_mm'].idxmax()]
drier = wx.loc[wx['precip_mm'].idxmin()]

# Subtract to find how much wetter the wetter date was.
difference = wettest['precip_mm'] - drier['precip_mm']

if difference == 0:
    print('September 5 and October 3 had the same precipitation.')
else:
    print(
        f"{wettest['date'].strftime('%Y-%m-%d')} was wetter than "
        f"{drier['date'].strftime('%Y-%m-%d')} by "
        f"{difference:.1f} mm of precipitation."
    )

2026-10-03 was wetter than 2026-09-05 by 12.2 mm of precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [19]:
# Record when this cell runs so the data source can be documented.
pulled_at = pd.Timestamp.now(tz='America/New_York').strftime(
    '%Y-%m-%d %H:%M:%S %Z'
)

# State clearly whether these rows came from the API or the sample.
if used_fallback:
    data_source = 'fallback sample'
else:
    data_source = 'Open-Meteo historical archive API'

print('Source:', data_source)
print('Endpoint:', BASE)
print('Coordinates: latitude 38.03, longitude -78.51')
print('Date range requested: 2026-09-05 through 2026-10-03')
print('Pulled:', pulled_at)

Source: Open-Meteo historical archive API
Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: latitude 38.03, longitude -78.51
Date range requested: 2026-09-05 through 2026-10-03
Pulled: 2026-10-11 15:51:02 EDT


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

Assuming the empty DataFrame still has the date and weather columns, a left merge would keep all 200 sales rows but leave the weather values missing. A chart might look blank because could skip missing values. If someone fills the missing weather values with zero, the chart could falsely show that there was no rain. The merge may result in 200 rows, so checking for missing weather matches is important.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [20]:
# Charlottesville data comes from wx, including its source label.
charlottesville = wx.copy()
charlottesville['city'] = 'Charlottesville'

# Request the same date range for Orlando.
orlando_raw = get_weather(
    '2026-09-05',
    '2026-10-03',
    lat=28.54,
    lon=-81.38
)

# Keep only the two dates we are comparing.
orlando_raw = orlando_raw[
    orlando_raw['time'].isin(dates_to_keep)
].copy()

# Rename the API columns and add city and source labels.
orlando = orlando_raw.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
orlando['date'] = pd.to_datetime(orlando['date'])
orlando['source'] = 'open-meteo'
orlando['city'] = 'Orlando'

# Stack both cities into one long-format table.
both_cities = pd.concat(
    [charlottesville, orlando],
    ignore_index=True
)

both_cities = both_cities[
    ['date', 'city', 'temp_max', 'precip_mm', 'source']
]

both_cities

,date,city,temp_max,precip_mm,source
0,2026-09-05,Charlottesville,29.4,2.5,open-meteo
1,2026-10-03,Charlottesville,22.0,14.7,open-meteo
2,2026-09-05,Orlando,32.1,3.9,open-meteo
3,2026-10-03,Orlando,30.4,14.5,open-meteo
